# 02 - Train Personalized User Model

Parameterized notebook that trains a SKLearn RandomForestClassifier to predict `repeat_purchase` for a given user.
- Accepts `user_id` as a parameter (e.g., `user_alice`, `user_bob`, etc.)
- Derives table and model names dynamically
- Feature engineering with categorical encoding and numeric scaling
- MLflow experiment tracking with full metrics
- Model registered to Unity Catalog

**Called in parallel by LakeFlow Job** — one task per user with different `user_id` parameter values.

In [0]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from mlflow.models import infer_signature

# Parameters
dbutils.widgets.text("user_id", "user_alice", "User ID")
dbutils.widgets.text("catalog", "custom_ml", "Catalog")  # Only hardcoded default
dbutils.widgets.text("schema", "hyper_personalization", "Schema")

USER_ID = dbutils.widgets.get("user_id")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

# Derived names
TABLE_NAME = f"{CATALOG}.{SCHEMA}.{USER_ID}_purchases"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.{USER_ID}_model"

mlflow.set_registry_uri("databricks-uc")
print(f"Training model for: {USER_ID}")
print(f"Reading from table: {TABLE_NAME}")
print(f"Model will be registered as: {MODEL_NAME}")

In [0]:
# Load user-specific purchase data
df = spark.table(TABLE_NAME).toPandas()
print(f"Loaded {len(df)} records for {USER_ID}")
print(f"Target distribution:\n{df['repeat_purchase'].value_counts()}")
print(f"\nRepeat purchase rate: {df['repeat_purchase'].mean():.2%}")

In [0]:
# Feature Engineering
# Encode product_name as numeric
le_product = LabelEncoder()
df["product_encoded"] = le_product.fit_transform(df["product_name"])

# Extract temporal features from purchase_date
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
df["day_of_week"] = df["purchase_date"].dt.dayofweek
df["month"] = df["purchase_date"].dt.month

# Define features and target
feature_cols = ["product_encoded", "price", "quantity", "rating", "day_of_week", "month"]
X = df[feature_cols].values
y = df["repeat_purchase"].values

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Features: {feature_cols}")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [0]:
# Train model with MLflow tracking
with mlflow.start_run(run_name=f"{USER_ID}_training") as run:
    # Hyperparameters
    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "random_state": 42,
    }
    mlflow.log_params(params)
    mlflow.log_param("user_id", USER_ID)
    mlflow.log_param("source_table", TABLE_NAME)
    mlflow.log_param("features", feature_cols)
    mlflow.log_param("num_training_records", len(X_train))
    
    # Train
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("auc_roc", auc)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC:  {auc:.4f}")
    
    # Log and register model with signature
    signature = infer_signature(X_train, model.predict(X_train))
    
    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=X_train[:5],
        registered_model_name=MODEL_NAME,
    )
    
    print(f"\nModel registered as: {MODEL_NAME}")
    print(f"Run ID: {run.info.run_id}")